# Penguins Species Classifier

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Configuration

In [3]:
DATA_PATH = "data/penguins.csv"
FEATURE_COLS = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "bill_ratio",
    "body_mass_scaled",
]
TARGET_COL = "species"

RANDOM_SEED = 42
TEST_SIZE = 0.2

MODEL_CONFIG = [
    ("Random Forest", RandomForestClassifier, {"n_estimators": 100, "random_state": RANDOM_SEED}),
    ("Logistic Regression", LogisticRegression, {"max_iter": 10000, "random_state": RANDOM_SEED}),
    ("Gradient Boosting", GradientBoostingClassifier, {"random_state": RANDOM_SEED}),
]

## Function Definitions

In [4]:
def load_data(path):
    df = pd.read_csv(path)

    print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    return df

In [5]:
def clean_data(df):
    df_clean = df.dropna()

    print(f"Data cleaned: {df.shape[0]} → {df_clean.shape[0]} rows")
    return df_clean

In [6]:
def encode_categoricals(df):

    # Convert text labels to numbers for machine learning.

    #   Many ML models only understand numbers, so we convert:
    #     "Adelie" → 0, "Chinstrap" → 1, "Gentoo" → 2, etc.

    #   We save the encoders so we can convert numbers back to text later.

    df = df.copy()

    # Create a separate encoder for each text column
    species_encoder = LabelEncoder()
    island_encoder = LabelEncoder()
    sex_encoder = LabelEncoder()

    # Apply each encoder to its column
    df["species"] = species_encoder.fit_transform(df["species"])
    df["island"] = island_encoder.fit_transform(df["island"])
    df["sex"] = sex_encoder.fit_transform(df["sex"])

    # Save encoders in a dictionary so we can use them later
    encoders = {
        "species": species_encoder,
        "island": island_encoder,
        "sex": sex_encoder,
    }

    return df, encoders

In [7]:
def add_engineered_features(df):
    df = df.copy()
    df["bill_ratio"] = df["bill_length_mm"] / df["bill_depth_mm"]
    df["body_mass_scaled"] = df["body_mass_g"] / 1000
    print("Feature engineering completed")
    return df

In [8]:
def scale_features(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    print("Feature scaling completed")
    return X_train_scaled, X_test_scaled, scaler

In [9]:
def build_features(df):
    df, encoders = encode_categoricals(df)
    df = add_engineered_features(df)
    print("Feature building completed")
    return df, encoders

In [10]:
def train_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    print("Model training completed")

In [11]:
def train_all_models(model_config, X_train, y_train):
    """Train all models from configuration.
    
    Takes a list of (name, ModelClass, hyperparams) tuples,
    instantiates and trains each model, prints progress.
    
    Returns a dictionary of trained models: {"model_name": model_instance, ...}
    """
    trained_models = {}
    
    for name, ModelClass, params in model_config:
        model = ModelClass(**params)
        train_model(model, X_train, y_train)
        trained_models[name] = model
        print(f"  ✓ {name} saved")
    
    print(f"\nAll {len(trained_models)} models trained successfully")
    return trained_models

In [12]:
def evaluate(model, X_test, y_test):
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    return predictions, accuracy

In [13]:
def evaluate_models(models, y_test):
    results = []
    for name, model, X_test_data in models:
        preds, acc = evaluate(model, X_test_data, y_test)
        results.append((name, model, acc, preds))
        print(f"{name:25} accuracy: {acc:.4f}")

    best_name, best_model, best_acc, best_preds = max(results, key=lambda x: x[2])
    print(f"\nBest model: {best_name} ({best_acc:.4f})")

    return results, best_name, best_model, best_acc, best_preds

In [14]:
def predict_species(sample, model, scaler, encoders, feature_cols):
    """Predict species for a new sample.

    Takes raw sample data, applies feature engineering and scaling,
    then makes a prediction and decodes the result back to species name.
    """
    sample_df = pd.DataFrame([sample])
    sample_df = add_engineered_features(sample_df)
    sample_df = sample_df[feature_cols]
    sample_scaled = scaler.transform(sample_df)

    prediction = model.predict(sample_scaled)[0]
    species_name = encoders["species"].inverse_transform([prediction])[0]

    print(f"Prediction completed: {species_name}")
    return prediction, species_name

## Pipeline

In [15]:
df = load_data(DATA_PATH)
df = clean_data(df)
df, encoders = build_features(df)

Data loaded: 344 rows, 7 columns
Data cleaned: 344 → 333 rows
Feature engineering completed
Feature building completed


In [16]:
X = df[FEATURE_COLS]
y = df[TARGET_COL]

In [17]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

# Scale features
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Feature scaling completed
Training set size: 266 samples
Test set size: 67 samples


## Training

In [18]:
# Train all models from configuration
trained_models = train_all_models(MODEL_CONFIG, X_train_scaled, y_train)

Model training completed
  ✓ Random Forest saved


Model training completed
  ✓ Logistic Regression saved
Model training completed
  ✓ Gradient Boosting saved

All 3 models trained successfully


## Evaluation

In [19]:
models = [(name, model, X_test_scaled) for name, model in trained_models.items()]

results, best_name, best_model, best_acc, best_preds = evaluate_models(models, y_test)

Random Forest             accuracy: 1.0000
Logistic Regression       accuracy: 1.0000
Gradient Boosting         accuracy: 1.0000

Best model: Random Forest (1.0000)


## Predict on a New Sample

In [20]:
# Create sample data
sample = {
    "bill_length_mm": 39.5,
    "bill_depth_mm": 17.4,
    "flipper_length_mm": 187,
    "body_mass_g": 3600,
}

In [21]:
prediction, species_name = predict_species(
    sample, best_model, scaler, encoders, FEATURE_COLS
)

Feature engineering completed
Prediction completed: Adelie


In [22]:
print(f"Using {best_name} model:")
print("Predicted species label:", prediction)
print("Decoded species:", species_name)

Using Random Forest model:
Predicted species label: 0
Decoded species: Adelie
